In [15]:
import os
from dotenv import load_dotenv
from google import genai
from elasticsearch import Elasticsearch



In [16]:
# Load variables and clients
load_dotenv()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
# Connect to our persistent Docker storage index from the last module
es_client = Elasticsearch("http://localhost:9200")



In [ ]:
def search_faq_database(query: str) -> list:
    """
    Searches the course FAQ database via Elasticsearch to find relevant answers 
    regarding enrollment, deadlines, certificates, and technical prerequisites.
    """
    # Simplified full-text search layout matching modern v8 syntax
    response = es_client.search(
        index="course-questions",
        query={
            "multi_match": {
                "query": query,
                "fields": ["question^2", "text"],
                "type": "best_fields"
            }
        },
        size=3
    )
    
    
    results = []
    for hit in response['hits']['hits']:
        results.append(hit['_source'])
    return results

In [18]:
# Pass the function directly into the tools list parameter array
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hey, I just found out about the course. Am I still allowed to enroll and join now?",
    config=dict(
        tools=[search_faq_database] # Registering our local search method
    )
)

# Look at how the agent decided to respond
print("Final Answer:\n", response.text)

Final Answer:
 While you might be late for the first couple of homeworks, they are not required for the certificate. To earn the certificate, you need to pass a project and evaluate 3 of your peers' projects, which starts later in the course. So, you can still join!


In [19]:
# Check the execution trace history to see the function call metadata
for part in response.candidates[0].content.parts:
    if part.function_call:
        print(f"🤖 Agent decided to execute tool: {part.function_call.name}")
        print(f"📥 Extracted search arguments: {part.function_call.args}")

In [20]:
# 1. Create a stateful chat session with your tool registered
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=dict(tools=[search_faq_database])
)

# 2. Send the message through the chat session
response = chat.send_message("Hey, I just found out about the course. Am I still allowed to enroll and join now?")

print("Final Answer:\n", response.text)

print("\n" + "="*40 + "\n🤖 INSPECTING THE AGENT TRACE:\n" + "="*40)

# 3. Now we can cleanly loop through the conversation history log!
for message in chat.get_history():
    print(f"\n--- Role: {message.role.upper()} ---")
    for part in message.parts:
        if part.text:
            print(f"📝 Text: {part.text.strip()}")
        if part.function_call:
            print(f"⚙️ Called Tool: {part.function_call.name}")
            print(f"📥 Arguments Sent: {part.function_call.args}")
        if part.function_response:
            print(f"📤 Database Returned Data: {part.function_response.response}")

Final Answer:
 It appears that enrollment deadlines are published per cohort. You can find them in the current cohort's folder in the course repo, on the course website, or in the course Google Calendar. There might also be announcements on Slack regarding extensions.

To give you a more precise answer, could you please tell me which course you are interested in?

🤖 INSPECTING THE AGENT TRACE:

--- Role: USER ---
📝 Text: Hey, I just found out about the course. Am I still allowed to enroll and join now?

--- Role: MODEL ---
⚙️ Called Tool: search_faq_database
📥 Arguments Sent: {'query': 'enrollment deadlines'}

--- Role: USER ---
📤 Database Returned Data: {'result': [{'id': 'd3f485cd10', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Homework: What are homework and project deadlines?', 'answer': "Deadlines are published per cohort. Find them in:\n\n- The current cohort's folder in the [course repo](https://github.com/DataTalksClub/data

In [24]:
# 1. Create a stateful chat session with the tool enabled
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=dict(
        tools=[search_faq_database]  # The SDK registers this function
    )
)

user_message = "Hey! Can I still register for the course, and do I need to know PySpark before starting?"
print(f"👤 User: {user_message}\n")

# 2. Send the message. The SDK natively resolves the function call, executes 
#    'search_faq_database' locally, sends the logs back, and returns the grounded answer.
response = chat.send_message(user_message)

print(f"✨ Final Answer:\n{response.text}")

👤 User: Hey! Can I still register for the course, and do I need to know PySpark before starting?

✨ Final Answer:
Regarding your questions:

You don't need to formally enroll to submit homework; you can just log into the homework form when it opens. The Airtable registration is primarily for announcements.

The FAQ doesn't specifically mention PySpark as a prerequisite. However, it does state that knowing time series is "helpful but not necessary" for the 'Predicting Financial Time-Series' workshop. While this isn't directly about PySpark, it suggests that some related knowledge might be beneficial but not strictly required, and provides guidance on how to pick up necessary skills.


In [25]:
user_message = "Hey! Can I still register for the course, and do I need to know PySpark before starting?"
print(f"👤 User: {user_message}\n")

# Start the continuous autonomous loop
current_prompt = user_message
max_turns = 5  # Stop condition guardrail to prevent infinite API calls
turn_count = 0
active = True


👤 User: Hey! Can I still register for the course, and do I need to know PySpark before starting?



In [26]:

while active and turn_count < max_turns:
    turn_count += 1
    print(f"🔄 --- AGENT LOOP TURN {turn_count} ---")
    
    # Send the current prompt or tool response back to the chat history state
    response = chat.send_message(current_prompt)
    
    # Track if the model wanted to execute a tool in this turn
    tool_called = False
    
    # Inspect what the agent decided to do
    for part in response.candidates[0].content.parts:
        # Scenario A: The Agent decides it needs to execute an Action/Tool
        if part.function_call:
            tool_called = True
            tool_name = part.function_call.name
            tool_args = part.function_call.args
            
            print(f"🤖 Thought: I need more context. Executing action...")
            print(f"⚙️ Action: Calling '{tool_name}' with arguments: {dict(tool_args)}")
            
            # Dynamically execute our local function using the arguments generated by the LLM
            if tool_name == "search_faq_database":
                # Extract the query string safely
                search_query = tool_args.get("query")
                
                # Run the search against your local Elasticsearch container
                observation_result = search_faq_database(query=search_query)
                
                print(f"📥 Observation: Retrieved {len(observation_result)} raw matches from Elasticsearch.")
                
                # Prepare the observation payload to send back to Gemini in the next iteration
                current_prompt = observation_result
            else:
                current_prompt = f"Error: Tool '{tool_name}' is not recognized."
                
        # Scenario B: The Agent decides it has completed its task and outputs text
        elif part.text and not tool_called:
            print("🤖 Thought: I have collected all the required details to compile my final answer.")
            print(f"\n✨ Final Answer:\n{part.text}")
            active = False  # Break the loop safely
            
    if turn_count == max_turns and active:
        print("⚠️ Guardrail reached: Stopped agent execution loop to prevent an infinite run.")

🔄 --- AGENT LOOP TURN 1 ---
🤖 Thought: I have collected all the required details to compile my final answer.

✨ Final Answer:
Regarding your questions:

You don't need to formally enroll to submit homework; you can just log into the homework form when it opens. The Airtable registration is primarily for announcements.

The FAQ doesn't specifically mention PySpark as a prerequisite. However, it does state that knowing time series is "helpful but not necessary" for the 'Predicting Financial Time-Series' workshop. While this isn't directly about PySpark, it suggests that some related knowledge might be beneficial but not strictly required, and provides guidance on how to pick up necessary skills.
